![NVIDIA Header](../../shared/assets/nvidia_header.png)

# Daily Financial Briefing Agent

### A multi-API skill with a custom "Wall Street analyst" personality

⏱ Estimated time: 40 minutes

## Learning Objectives

By the end of this notebook, you will be able to:

<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Review a canonical multi-API <a href="https://docs.openclaw.ai/tools/creating-skills">SKILL.md</a> that composes five services</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Apply a canonical financial-analyst <a href="https://docs.openclaw.ai/concepts/soul">SOUL.md</a> to one verified agent</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Add and validate least-privilege network entries for multiple public APIs</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Separate live source evidence, persona framing, and scheduled-delivery configuration</p>

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2024-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

## Setup

**Platform:** Validated NemoClaw host (Ubuntu 24.04, DGX Spark, or Apple Silicon macOS with Colima)

**Prerequisites:** Complete **[NemoClaw with OpenClaw](../03-nemoclaw/01_nemoclaw_openclaw.ipynb)**, finish the **[NASA APOD](01_nasa_apod.ipynb)** exercise, and confirm the exact `agent` sandbox before changing policy or deploying skills.

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd()
for _parent in [_cwd, *_cwd.parents]:
    if (_parent / "pyproject.toml").exists():
        if str(_parent) not in sys.path:
            sys.path.insert(0, str(_parent))
        break

In [ ]:
from shared.utils.setup import setup

setup(
    require_docker=True,
    require_node=True,
    require_openshell=True,
    required_keys=[],
)

In [ ]:
import importlib
import json
import time
from pathlib import Path

import shared.utils.platform as _plat_mod

importlib.reload(_plat_mod)

from shared.utils.display import run_command
from shared.utils.mutations import authorize_mutation
from shared.utils.platform import (
    create_and_onboard_agent,
    deploy_skill_to_agent,
    get_dashboard_access_note,
    get_dashboard_access_url,
    list_openclaw_agent_ids,
)

MUTATION_SANDBOX = "agent"

analyst_soul_path = Path("souls/financial-analyst.md")
if not analyst_soul_path.is_file():
    raise FileNotFoundError(f"Canonical analyst SOUL is missing: {analyst_soul_path}")
analyst_soul = analyst_soul_path.read_text(encoding="utf-8")

analyst_agent_preexisting = "financial-analyst" in list_openclaw_agent_ids(MUTATION_SANDBOX)
analyst_agent_ready = False
course_agent_created = False
agent_reconciliation_allowed = analyst_agent_preexisting
if not agent_reconciliation_allowed:
    agent_reconciliation_allowed = authorize_mutation(
        action=("Create the course-owned financial-analyst agent from its canonical SOUL.md"),
        target=MUTATION_SANDBOX,
    )

if agent_reconciliation_allowed:
    creation_ok, detail = create_and_onboard_agent(
        "financial-analyst",
        name="Market Brief",
        emoji="📊",
        soul_md=analyst_soul,
        sandbox=MUTATION_SANDBOX,
        update_existing=False,
    )
    print(f"Agent: {detail}")
    analyst_agent_ready = creation_ok
    course_agent_created = creation_ok and not analyst_agent_preexisting
    if not creation_ok:
        print("SAFE SKIP: existing or partial agent state was preserved; later deployment skips.")
else:
    print("SAFE MODE: financial-analyst was absent and was not created.")

dashboard_url = get_dashboard_access_url()
dashboard_note = get_dashboard_access_note()
print(f"Dashboard: {dashboard_url}")
print(dashboard_note)


def run_briefing_api_probe(
    label: str,
    cmd: str,
    validator,
    *,
    timeout: int = 20,
    attempts: int = 3,
    pause: int = 5,
) -> object | None:
    """Run an authorized briefing API probe and require usable JSON."""
    if not globals().get("briefing_policy_ready", False):
        print(f"SAFE SKIP: {label} needs the verified briefing network policy.")
        return None
    last_output = ""
    for attempt in range(1, attempts + 1):
        print(f"{label}: attempt {attempt}/{attempts}")
        result = run_command(
            cmd,
            timeout=timeout,
            format_json=True,
            return_result=True,
        )
        last_output = f"{result.stdout}\n{result.stderr}".strip()
        if result.returncode == 0:
            try:
                payload = json.loads(result.stdout)
            except json.JSONDecodeError:
                print(f"{label} response was not valid JSON; retrying.")
            else:
                try:
                    valid = bool(validator(payload))
                except (AttributeError, KeyError, TypeError, ValueError) as exc:
                    valid = False
                    print(f"{label} validator rejected a malformed shape: {exc}")
                if valid:
                    print(f"{label}_OK")
                    return payload
                print(f"{label} JSON shape did not match expectations; retrying.")
        else:
            print(f"{label} failed with exit code {result.returncode}; retrying.")
        if attempt < attempts:
            time.sleep(pause)
    raise RuntimeError(
        f"{label} did not return usable JSON after {attempts} attempts. "
        f"Last output:\n{last_output[:1000]}"
    )

---

## 1. What We're Building

A **morning financial briefing agent** that:

- Calls **5 live APIs** for real market, news, weather, and holiday data
- Presents the data with a **"Wall Street analyst" personality** ([SOUL.md](https://docs.openclaw.ai/concepts/soul))
- Follows a **structured briefing format** every time
- Uses **zero credentials** — every API is free and auth-free

**Multi-API composition** means a single [skill](https://docs.openclaw.ai/tools/creating-skills) calls several independent services
and synthesizes their responses into one unified output. The agent orchestrates
sources in sequence, handles partial failures, and merges results — all declared in
the SKILL.md.

### The APIs

| API | What it provides | Auth | Endpoint |
|-----|-----------------|------|----------|
| **CoinGecko** | Bitcoin, Ethereum prices + 24hr change | None | `api.coingecko.com` |
| **Frankfurter** | USD→EUR, GBP, JPY exchange rates | None | `api.frankfurter.dev` |
| **Hacker News** | Top 5 tech stories | None | `hacker-news.firebaseio.com` |
| **Open-Meteo** | Weather for a configurable city | None | `api.open-meteo.com` |
| **Nager.Date** | Upcoming US public holidays | None | `date.nager.at` |

### What Makes This Impressive

- **Multi-source composition** — one skill calls 5 APIs and synthesizes results
- **Custom personality** — the SOUL.md presents data like a financial analyst
- **Real live data** — actual Bitcoin prices, exchange rates, and Hacker News stories
- **Repeatable format** — every run follows the same structure


---

## 2. Inspect the Canonical SOUL.md

[SOUL.md](https://docs.openclaw.ai/concepts/soul) defines an agent's tone and behavioral boundaries. This lesson keeps one checked-in persona as the source of truth. The setup cell reads those exact bytes for agent verification or creation; running the notebook never regenerates the file.

In [ ]:
analyst_skill_path = Path("skills/daily-briefing/SKILL.md")
assert analyst_soul_path.is_file(), f"Missing canonical SOUL: {analyst_soul_path}"
assert analyst_skill_path.is_file(), f"Missing canonical skill: {analyst_skill_path}"
print(f"Canonical SOUL: {analyst_soul_path}")
print(f"Canonical skill: {analyst_skill_path}")

Display the exact financial-analyst persona used during setup. **Expected result:** the cell confirms the file has not changed since setup and prints its reporting boundaries. No course source file is written.

In [ ]:
displayed_analyst_soul = analyst_soul_path.read_text(encoding="utf-8")
assert displayed_analyst_soul == analyst_soul, "Canonical SOUL changed during this run"
print(displayed_analyst_soul)

Compare this to the default [SOUL.md](https://docs.openclaw.ai/concepts/soul) that ships with OpenClaw — the default says "Have opinions,"
"Be genuinely helpful," and "Remember you're a guest." This persona says "Lead with
numbers," "Flag anomalies," and "Never speculate." Same framework, different agent
behavior.


---

## 3. Inspect the Canonical Multi-API SKILL.md

The checked-in skill is the deployable artifact. Validate and display it instead of rebuilding a second copy inside the notebook.

In [ ]:
analyst_skill_text = analyst_skill_path.read_text(encoding="utf-8")
analyst_skill_lines = analyst_skill_text.splitlines()
assert analyst_skill_lines[0] == "---", "SKILL.md frontmatter must begin on line 1"
closing_frontmatter = analyst_skill_lines.index("---", 1)
assert analyst_skill_lines[closing_frontmatter + 1] == "<!--"
assert "SPDX-License-Identifier: Apache-2.0" in "\n".join(
    analyst_skill_lines[closing_frontmatter + 1 : closing_frontmatter + 7]
)
print(analyst_skill_text)

---

## 4. Add Network Policies for All 5 APIs

OpenShell [sandboxes](https://docs.nvidia.com/openshell/sandboxes/manage-sandboxes) deny all outbound network traffic by default. Each API
the agent calls needs an explicit
[network policy](https://docs.nvidia.com/openshell/reference/policy-schema) entry that
specifies the host, port, protocol, and which binaries may initiate the connection.
We add one entry per API below — `enforcement: enforce` blocks requests to unlisted
hosts, and `access: read-only` restricts the agent to GET requests.
See the full [policy customization guide](https://docs.nvidia.com/openshell/sandboxes/policies)
for additional options.


In [ ]:
from shared.utils.policies import load_base_policy, upsert_network_policies

briefing_apis = {
    "coingecko": {"host": "api.coingecko.com", "port": 443},
    "frankfurter": {"host": "api.frankfurter.dev", "port": 443},
    "hackernews": {"host": "hacker-news.firebaseio.com", "port": 443},
    "open_meteo": {"host": "api.open-meteo.com", "port": 443},
    "nager_date": {"host": "date.nager.at", "port": 443},
}

briefing_policy = {
    name: {
        "endpoints": [
            {
                "host": api["host"],
                "port": api["port"],
                "protocol": "rest",
                "enforcement": "enforce",
                "access": "read-only",
            }
        ],
        "binaries": [
            {"path": "/usr/bin/curl"},
            {"path": "/usr/local/bin/node"},
        ],
    }
    for name, api in briefing_apis.items()
}


def briefing_entry_matches(name: str, entry: object) -> bool:
    """Require the intended single endpoint and exact executable set."""
    if not isinstance(entry, dict) or entry.get("name", name) != name:
        return False
    endpoints = entry.get("endpoints")
    if not isinstance(endpoints, list) or len(endpoints) != 1:
        return False
    expected_endpoint = briefing_policy[name]["endpoints"][0]
    endpoint = endpoints[0]
    if not isinstance(endpoint, dict):
        return False
    if not all(endpoint.get(key) == value for key, value in expected_endpoint.items()):
        return False
    binaries = entry.get("binaries")
    if not isinstance(binaries, list):
        return False
    actual_paths = {
        item.get("path") for item in binaries if isinstance(item, dict) and item.get("path")
    }
    expected_paths = {item["path"] for item in briefing_policy[name]["binaries"]}
    return actual_paths == expected_paths


def classify_briefing_entries(policy: dict) -> tuple[set[str], set[str], set[str]]:
    network = policy.get("network_policies", {})
    if not isinstance(network, dict):
        raise RuntimeError("network_policies must be a mapping")
    matching = {name for name in briefing_policy if briefing_entry_matches(name, network.get(name))}
    present = {name for name in briefing_policy if name in network}
    return matching, set(briefing_policy) - present, present - matching


# Read immediately before the decision. Existing names are preserved, and only
# missing course entries may be added after exact target-scoped authorization.
policy = load_base_policy(MUTATION_SANDBOX)
matching, missing, conflicting = classify_briefing_entries(policy)
briefing_policy_ready = False
course_policy_added = False

if conflicting:
    print(
        f"PRESERVED: incompatible existing briefing entries were not changed: {sorted(conflicting)}"
    )
    print("SAFE SKIP: live briefing probes require operator review of those entries.")
elif not missing:
    briefing_policy_ready = True
    print("PRESERVED: all required briefing policy entries already match the course shape.")
elif authorize_mutation(
    action=f"Add missing course briefing network policies: {sorted(missing)}",
    target=MUTATION_SANDBOX,
):
    try:
        policy = upsert_network_policies(
            MUTATION_SANDBOX,
            {name: briefing_policy[name] for name in missing},
            timeout=30,
            replace_existing=False,
        )
    except RuntimeError:
        # A concurrent/rerun entry is never overwritten. Re-read before deciding
        # whether the exact course capability is now available.
        policy = load_base_policy(MUTATION_SANDBOX)
        matching, missing, conflicting = classify_briefing_entries(policy)
        if conflicting:
            print(f"PRESERVED: concurrent entries were not replaced: {sorted(conflicting)}")
        elif missing:
            raise
        else:
            briefing_policy_ready = True
            print("PRESERVED: another writer added the exact entries before this apply.")
    else:
        matching, missing, conflicting = classify_briefing_entries(policy)
        if missing or conflicting:
            raise RuntimeError("Applied briefing policy did not retain the requested narrow shape")
        briefing_policy_ready = True
        course_policy_added = True
        print(f"ADDED: {len(matching)} course briefing policy entries are ready")
else:
    print("SAFE MODE: missing briefing rules were not added; live API probes will skip.")

print(f"Briefing policy ready for live course probes: {briefing_policy_ready}")

---

## 5. Test Each API from Inside the Sandbox

Before deploying the [skill](https://docs.openclaw.ai/tools/creating-skills), verify each API is reachable through the [network policy](https://docs.nvidia.com/openshell/reference/policy-schema).
This is the raw data the agent will synthesize into a briefing.


In [ ]:
# Crypto prices: Bitcoin, Ethereum, and Solana with 24-hour change.
print("--- CoinGecko: Crypto Prices ---")
crypto_payload = run_briefing_api_probe(
    "COINGECKO",
    f"openshell sandbox exec -n {MUTATION_SANDBOX} --timeout 15 -- "
    "curl -fsS --max-time 15 'https://api.coingecko.com/api/v3/simple/price?ids=bitcoin,ethereum,solana&vs_currencies=usd&include_24hr_change=true'",
    lambda payload: (
        isinstance(payload, dict)
        and all(
            isinstance(payload.get(asset), dict)
            and isinstance(payload[asset].get("usd"), (int, float))
            for asset in ("bitcoin", "ethereum", "solana")
        )
    ),
)

Query the deterministic USD exchange-rate feed for the briefing. **Expected result:** the validated response contains EUR, GBP, JPY, and CNY rates, although their numeric values will vary. This matters because the analyst needs structured currency evidence rather than invented market context.

In [ ]:
# Exchange rates: USD to EUR, GBP, JPY, and CNY.
print("--- Frankfurter: Exchange Rates ---")
forex_payload = run_briefing_api_probe(
    "FRANKFURTER",
    f"openshell sandbox exec -n {MUTATION_SANDBOX} --timeout 15 -- "
    "curl -fsSL --max-time 15 'https://api.frankfurter.dev/v1/latest?from=USD&to=EUR,GBP,JPY,CNY'",
    lambda payload: (
        isinstance(payload, dict)
        and isinstance(payload.get("rates"), dict)
        and all(
            isinstance(payload["rates"].get(currency), (int, float))
            for currency in ("EUR", "GBP", "JPY", "CNY")
        )
    ),
)

Fetch the current Hacker News story identifiers for the technology pulse. **Expected result:** the response is a list with at least five integer IDs. This matters because live story IDs provide a verifiable starting point for later article selection.

In [ ]:
# Hacker News: current top-story identifiers.
print("--- Hacker News: Top Story IDs ---")
hn_payload = run_briefing_api_probe(
    "HACKERNEWS",
    f"openshell sandbox exec -n {MUTATION_SANDBOX} --timeout 15 -- "
    "curl -fsS --max-time 15 'https://hacker-news.firebaseio.com/v0/topstories.json'",
    lambda payload: (
        isinstance(payload, list)
        and len(payload) >= 5
        and all(isinstance(item, int) and item > 0 for item in payload[:5])
    ),
)

Fetch current weather at NVIDIA's Santa Clara location. **Expected result:** the response includes a `current` object with temperature data, while the actual reading may change. This matters because the briefing demonstrates how one agent can combine market and operational context.

In [ ]:
# Weather: Santa Clara, California (NVIDIA headquarters).
print("--- Open-Meteo: Weather ---")
weather_payload = run_briefing_api_probe(
    "OPEN_METEO",
    f"openshell sandbox exec -n {MUTATION_SANDBOX} --timeout 15 -- "
    "curl -fsS --max-time 15 'https://api.open-meteo.com/v1/forecast?latitude=37.39&longitude=-122.08&current=temperature_2m,wind_speed_10m,relative_humidity_2m&temperature_unit=fahrenheit'",
    lambda payload: (
        isinstance(payload, dict)
        and isinstance(payload.get("current"), dict)
        and isinstance(payload["current"].get("temperature_2m"), (int, float))
    ),
)

Fetch the next US public holidays for the upcoming-events section. **Expected result:** the response contains at least one entry with a date and name. This matters because validating the schema prevents a successful HTTP response from masquerading as useful briefing data.

In [ ]:
# Upcoming United States public holidays.
print("--- Nager.Date: Next US Holidays ---")
holiday_payload = run_briefing_api_probe(
    "NAGER_DATE",
    f"openshell sandbox exec -n {MUTATION_SANDBOX} --timeout 15 -- "
    "curl -fsS --max-time 15 'https://date.nager.at/api/v3/NextPublicHolidays/US'",
    lambda payload: (
        isinstance(payload, list)
        and bool(payload)
        and isinstance(payload[0], dict)
        and all(
            isinstance(payload[0].get(field), str) and payload[0][field].strip()
            for field in ("date", "name")
        )
    ),
)

When `briefing_policy_ready` is true and each payload variable contains a validated object, the five probes provide live market, exchange-rate, story, weather, and holiday evidence. In safe mode or after a failed probe, `None` is not presented as live data; inspect the per-cell result before claiming a complete briefing input set.

---

## 6. Deploy the Skill to the Market Brief Agent

OpenClaw 2026.6.10 installs local directories for an exact agent with its native CLI:

```bash
openshell sandbox exec -n agent --no-tty -- \
  openclaw skills install /sandbox/path/to/daily-briefing --agent financial-analyst
```

The course helper uploads the canonical source to unique staging, preserves differing existing content unless overwrite is explicitly requested, invokes this supported command, verifies the selected agent's skill, and cleans up staging. `--global` is a separate, explicit shared-scope choice.

In [ ]:
# Deploy only to a verified agent and only after target-scoped authorization.
analyst_skill_ready = False
if not analyst_agent_ready:
    print("SAFE SKIP: financial-analyst profile did not match; no skill was deployed.")
elif authorize_mutation(
    action="Deploy or preserve the financial-analyst exact-agent skill",
    target=MUTATION_SANDBOX,
):
    ok, detail = deploy_skill_to_agent(
        "financial-analyst",
        "skills/daily-briefing/",
        sandbox=MUTATION_SANDBOX,
        overwrite_existing=False,
        remove_global=False,
    )
    print(detail)
    if ok:
        analyst_skill_ready = True
    else:
        print("SAFE SKIP: conflicting workspace skill was preserved; readiness is false.")
else:
    print("SAFE MODE: workspace and global skills were unchanged.")

Assert that the briefing skill was deployed to the financial-analyst workspace. **Expected result:** the helper reports success and the cell confirms agent-local scope. This matters because the deployment should not alter unrelated agent personas.

In [ ]:
# Report the observed deployment outcome without turning safe mode into a false failure.
if analyst_skill_ready:
    assert ok, detail
    print(detail)
    print("The daily-briefing skill is scoped to the financial-analyst workspace.")
else:
    print("SAFE-SKIP_EVIDENCE: no live daily-briefing deployment was claimed.")

---

## 7. The Power of Personality

The canonical [SOUL.md](https://docs.openclaw.ai/concepts/soul) loaded for this agent supplies persistent financial-analyst role and presentation constraints. It does not make sampled model output deterministic; source data, tools, session context, and model behavior still affect each response without changing the skill definition. Here's how the **same data** looks with different personalities:

### Financial Analyst (our SOUL.md)
```
## Market Snapshot
- BTC: $74,244 (▼ 0.18%) — flat, no anomalies
- ETH: $2,330 (▼ 1.51%) — mild pullback, within normal range
- USD/EUR: 0.8480 | USD/JPY: 158.85

## Recommendations
1. No action needed on crypto — market is range-bound
2. Review the Hacker News AI compiler story for demo relevance
3. Memorial Day is in 40 days — schedule customer follow-ups before then
```

### Casual Tech Blogger (different SOUL.md)
```
Hey! Bitcoin's basically flat today at $74K — nothing exciting.
ETH dipped a bit. The really interesting thing is this Hacker News
story about... oh and it's going to be 50°F in Santa Clara, grab a jacket!
```

Same APIs, same data, completely different output. The personality file is the
differentiator — and it's just a Markdown file you can edit.


---

## 8. Scheduling the Briefing

OpenClaw 2026.6.10 accepts the cron expression and message as positional arguments. Run the supported CLI inside the exact sandbox, select the exact agent, isolate each scheduled transcript, and disable fallback delivery until an operator configures a destination:

```bash
openshell sandbox exec -n agent --no-tty -- \
  openclaw cron create --agent financial-analyst --session isolated --no-deliver \
  '30 7 * * 1-5' 'Prepare the weekday morning briefing'
```

The expression means 07:30 every Monday through Friday in the gateway's configured timezone. Review the created job with `openclaw cron list` before enabling any external delivery path.

---

## Try It Live

Target the exact Market Brief agent. If you need a clean transcript, reset the exact session through the host-side NemoClaw lifecycle command; do not send `/new` as if it were an ordinary agent prompt.

```bash
# Optional: replace <session-key> with the exact key shown in Sessions.
nemoclaw agent sessions reset '<session-key>' --agent financial-analyst

openshell sandbox exec -n agent --no-tty -- \
  openclaw agent --agent financial-analyst --message "Give me today's morning briefing"
```

Open the dashboard URL printed earlier. On DGX Spark, use the tokenized root URL over the workstation SSH tunnel, then inspect the Market Brief entry in the Dashboard **Sessions** view.

**Try these prompts:**

- *"Give me today's morning briefing"*
- *"What's happening in the crypto markets right now?"*
- *"Summarize the top Hacker News stories for me"*

> **Why `--agent financial-analyst`?** The daily-briefing skill belongs to that agent's workspace. Other agents should not inherit it implicitly.

---

## Real-World Connection

This pattern scales to any domain:

| Use case | APIs | Personality |
|----------|------|-------------|
| **Sales briefing** | CRM API, calendar, email | "Concise account manager" |
| **DevOps morning** | GitHub, PagerDuty, CloudWatch | "Terse ops engineer" |
| **Research digest** | arXiv, Semantic Scholar, HN | "Academic summarizer" |
| **Customer prep** | Salesforce, LinkedIn, news | "Customer success analyst" |

The [skill](https://docs.openclaw.ai/tools/creating-skills) provides the data sources. The [SOUL.md](https://docs.openclaw.ai/concepts/soul) provides the lens. The [network policy](https://docs.nvidia.com/openshell/reference/policy-schema)
controls what the agent reaches. All three pieces are visible, auditable, and
version-controllable.


---

## Checkpoint

Verify only evidence produced by your execution path.

1. **Check:** the canonical persona matched the exact agent profile.
   - **Expected:** `analyst_agent_ready` is true only after workspace, identity, and SOUL verification.
   - **Conflict:** existing state is preserved and deployment skips.

2. **Check:** every required API rule matched before live probes.
   - **Expected:** `briefing_policy_ready` is true and each non-`None` payload passed its total shape validator.
   - **Safe mode/conflict:** no complete live-input claim is made.

3. **Check:** the canonical skill was available to the exact agent.
   - **Expected:** an identical skill was preserved or installed with `--agent financial-analyst` after authorization.
   - **Conflict:** `analyst_skill_ready` remains false and differing content is preserved.

4. **Check:** an optional live agent response used current evidence.
   - **Expected:** only claim this after the agent and skill readiness flags are true and the response contains attributable values.
   - **If not working:** inspect each probe and `openclaw skills check --agent financial-analyst` inside the exact sandbox.

---

## ✅ What You Accomplished

<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Inspected canonical financial-analyst SOUL and daily-briefing skill sources without rewriting them</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Verified the exact agent profile before treating it as ready</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Used total JSON-shape validation for five public data sources</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Preserved incompatible policies and differing existing workspace skills</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Distinguished verified live evidence from conflict and safe-mode outcomes</p>
<p style="margin-left: 1.5em; margin-top: 0; margin-bottom: 0;">✅ &nbsp; Used the pinned exact-agent cron and session-lifecycle command surfaces</p>

## Next Steps

This is the last notebook in the distilled course. You went from the architecture, through OpenShell policy and the NemoClaw-managed OpenClaw runtime, to a policy-constrained agent that composes real APIs.

To go further, the full `claw-training-assets` course adds the Hermes and Deep Agents Code branches, OpenShell deep dives on enforcement testing and structured logs, and production operations: scheduling, backup, diagnostics, and recovery.

---

<small>

**Data Sources:** [CoinGecko API](https://www.coingecko.com/en/api) · [Frankfurter API](https://frankfurter.dev/) · [Hacker News API](https://github.com/HackerNews/API) · [Open-Meteo API](https://open-meteo.com/) · [Nager.Date API](https://date.nager.at/)

**OpenClaw Docs:** [Creating Skills](https://docs.openclaw.ai/tools/creating-skills) · [SOUL.md Concepts](https://docs.openclaw.ai/concepts/soul) · [Agent Workspaces](https://docs.openclaw.ai/concepts/agent-workspace) · [Multi-Agent](https://docs.openclaw.ai/concepts/multi-agent) · [Cron Scheduling](https://docs.openclaw.ai/automation/cron-jobs)

**OpenShell Docs:** [Network Policy Schema](https://docs.nvidia.com/openshell/reference/policy-schema) · [Policy Customization](https://docs.nvidia.com/openshell/sandboxes/policies) · [Security Best Practices](https://docs.nvidia.com/openshell/security/best-practices)

**NemoClaw Docs:** [CLI Command Reference](https://docs.nvidia.com/nemoclaw/latest/user-guide/openclaw/reference/commands)

</small>